In [50]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool

In [51]:
from langchain.agents import create_agent

In [ ]:
# LOAD ENV VARIABLES


os.environ["SSL_CERT_FILE"]=certifi.where()
load_dotenv()

GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY=os.getenv("WEATHER_API_KEY")


In [53]:
# Search Tool
search_tool=TavilySearchResults(max_results=2)
result=search_tool.invoke("Give me latest news on Indian politics")
result

[{'title': 'Live Politics News from India, Latest Politics headlines',
  'url': 'https://m.economictimes.com/news/politics',
  'content': "Odisha: BJD plans legal action, public demonstrations against Mines and Minerals (Amendment) BillOdisha: BJD plans legal action, public demonstrations against Mines and Minerals (Amendment) Bill\n India Inc backs Modi’s Independence Day push, vows to turn vision into actionIndia Inc backs Modi’s Independence Day push, vows to turn vision into action [...] BJP leader R Ashoka has demanded a white paper on Karnataka's guarantee schemes. He alleges that Chief Minister D K Shivakumar presented contradictory figures during his Independence Day address. Official records and the Chief Minister's Office have provided different beneficiary and expenditure numbers.\n\nBJP leader Biswajit Talukdar arrested for abducting businessman's son in West Bengal's Cooch Behar\n\n### BJP leader Biswajit Talukdar arrested for abducting businessman's son in West Bengal's C

In [54]:
# creaating a custom tool 
@tool
def get_weather_data(city:str)->str:
     """
    Fetch current weather information for a city.
    """
     url=( f"https://api.weatherstack.com/current?"
        f"access_key={WEATHER_API_KEY}&query={city}")
     
     response=requests.get(url) 
     data=response.json()
     if "current" not in data:
          return f"Could not fetch weather data for {city}"
     return (
           f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [55]:

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)
response = llm.invoke("Tell me an ai joke")
response

AIMessage(content=[{'type': 'text', 'text': 'Why did the AI cross the road?\n\nBecause it was trained to predict the next step, and it had seen 10 million pictures of chickens doing it.', 'extras': {'signature': 'EnEKbwERTTIPCwllrHWE0g4DPaY1wDJIEZErTzs44b/OZDSSXXWShd2/K7aP532MMRCUGxP3GiOMPfhMq2TIQC640kOzbVWfZMuRtzK4QRclalIpyi6TiOfR+WNIaLa3rAk95n3ym5WqghMWsAc5fz9Xfw=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a00c0f-ba4a-7030-a316-c1bfdfd70113-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 33, 'total_tokens': 39, 'input_token_details': {'cache_read': 0}})

In [56]:
tools=[search_tool,get_weather_data]

In [57]:
from langchain.agents import create_agent
# Create Agent
agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are a helpful ReAct-style AI agent.

When answering a question:
1. Determine what information is needed.
2. Decide which available tool is appropriate.
3. Use the tool when necessary.
4. Examine the tool result.
5. If additional information is needed, use another tool.
6. Once you have enough information, provide the final answer.

Always use the available tools when they are necessary to answer the question accurately.""",
debug=True
)
response = agent.invoke(
    {
        "messages": [
            ("user", "Find the capital of India and then find its current weather.")
        ]
    }
)

print(response["messages"][-1].content[0]["text"])

[values] {'messages': [HumanMessage(content='Find the capital of India and then find its current weather.', additional_kwargs={}, response_metadata={}, id='798e1c2a-ef1d-4762-b1fd-8023033c38ad')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "capital of India"}'}, '__gemini_function_call_thought_signatures__': {'call_3103581': 'EnEKbwERTTIPOgbvBTJI2xSVVydMxY6t8nKsXU7YbUkvKy3QD6J7kGM3VoV23q5GBwCYnF8Pw4X6a5STYdi3dLB2jJLYAHOxZOZfASBvwyFTlx9NXCQXZoXNIl2hrzkJFdZO4MD9yN8rk11dpgnAHqDANA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a00c0f-d55f-7081-ba4e-9482de8598c0-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'capital of India'}, 'id': 'call_3103581', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 241, 'output_tok